# Faadjou — entraînement ASR Wolof sur Kaggle

Chargez ces deux datasets Kaggle avant d’exécuter les cellules :
1. `faadjou-code` : dossier `faadjou-ai` sans les corpus complets.
2. `faadjou-wolof-subset` : contenu de `faadjou-ai/data/subset` uniquement.

In [ ]:
import os
from pathlib import Path

CODE = Path('/kaggle/input/faadjou-code')
DATA = Path('/kaggle/input/faadjou-wolof-subset')
print('Code:', CODE, 'existe=', CODE.exists())
print('Data:', DATA, 'existe=', DATA.exists())
print('Fichiers data:', [p.name for p in DATA.glob('*')])

In [ ]:
from pathlib import Path

CODE = Path('/kaggle/input/faadjou-code')
candidates = [CODE, CODE / 'faadjou-ai']
project_root = next((p for p in candidates if (p / 'pyproject.toml').exists()), None)
if project_root is None:
    raise FileNotFoundError("Dataset faadjou-code introuvable. Vérifiez le nom et la structure du dataset.")

!rm -rf /content/faadjou-ai
!cp -r {str(project_root)} /content/faadjou-ai
%cd /content/faadjou-ai
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt
!python -m pip install -e .

In [ ]:
!mkdir -p data/subset
!cp -r /kaggle/input/faadjou-wolof-subset/. data/subset/
!python -m json.tool data/subset/MANIFEST.json

In [ ]:
!python scripts/run_asr_eval.py --config configs/config.subset.yaml --max-samples 200 --batch-size 1

In [ ]:
!python scripts/finetune_asr.py --config configs/config.subset.yaml --max_steps 50 --output_dir /kaggle/working/checkpoints/whisper-wolof-ft

In [ ]:
from pathlib import Path
import shutil

export_dir = Path('/kaggle/working/faadjou-wolof-model')
export_dir.mkdir(parents=True, exist_ok=True)
final_model = Path('/kaggle/working/checkpoints/whisper-wolof-ft/final')
if final_model.exists():
    shutil.copytree(final_model, export_dir / 'final', dirs_exist_ok=True)
for name in ('asr_eval.csv', 'asr_finetune_compare.csv', 'asr_finetune_decision.txt'):
    src = Path('/content/faadjou-ai/results') / name
    if src.exists():
        shutil.copy2(src, export_dir / name)
shutil.make_archive('/kaggle/working/faadjou-wolof-model', 'zip', export_dir)
print('/kaggle/working/faadjou-wolof-model.zip')